In [16]:
%pip install mlflow azureml-mlflow

Note: you may need to restart the kernel to use updated packages.


In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

In [2]:
from azureml.core import Workspace, Experiment, Dataset
from azureml.core.model import Model
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

import mlflow

import pandas as pd
import numpy as np

import mltable
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

In [8]:
ml_client = MLClient.from_config(credential=DefaultAzureCredential())
data_asset = ml_client.data.get("bostonhousing", version="1")

tbl = mltable.load(f'azureml:/{data_asset.id}')

pddf_bh = tbl.to_pandas_dataframe()
display(pddf_bh.head(5))

# Your training code goes here
target = "medv"

X = pddf_bh.drop(target, axis=1)
y = pddf_bh[target]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Found the config file in: /config.json


,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,b,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


In [9]:
# Train the model
n_estimators = 100
model = RandomForestRegressor(n_estimators=n_estimators, random_state=42)
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")

Mean Squared Error: 7.901513892156864


In [14]:
ws = Workspace.from_config()
subscription_id = "08539d0e-620d-424c-aea8-56853ed8fe3e"
resource_group_name = "dspro2"
experiment_name = "test"

In [17]:
models = ml_client.models.list()
for model in models:
    print(model.name)

azureml_61b429ac-9ab5-43c6-a235-a7610a34e45b_output_mlflow_log_model_2069815210
model_bh
azureml_d7796f66-4182-4e26-9ad0-586cd6a49581_output_mlflow_log_model_510417770
azureml_16b95c13-7e59-49c5-aa33-d91c881106e0_output_mlflow_log_model_1395392451
azureml_8a73edcc-4fb3-4d36-9c96-06513fe53970_output_mlflow_log_model_2009320613


In [18]:
model_example = ml_client.models.get(name="azureml_61b429ac-9ab5-43c6-a235-a7610a34e45b_output_mlflow_log_model_2069815210", version="1")
print(model_example)

creation_context:
  created_at: '2025-02-26T11:05:55.679943+00:00'
  created_by: curdin derungs
  created_by_type: User
  last_modified_at: '2025-02-26T11:05:55.679943+00:00'
  last_modified_by: curdin derungs
  last_modified_by_type: User
flavors:
  python_function:
    env: "{\n  \"conda\": \"conda.yaml\",\n  \"virtualenv\": \"python_env.yaml\"\n\
      }"
    loader_module: mlflow.sklearn
    model_path: model.pkl
    predict_fn: predict
    python_version: 3.10.14
  sklearn:
    code: ''
    pickled_model: model.pkl
    serialization_format: cloudpickle
    sklearn_version: 1.5.2
id: azureml:/subscriptions/08539d0e-620d-424c-aea8-56853ed8fe3e/resourceGroups/dspro2/providers/Microsoft.MachineLearningServices/workspaces/dspro2ml/models/azureml_61b429ac-9ab5-43c6-a235-a7610a34e45b_output_mlflow_log_model_2069815210/versions/1
job_name: 61b429ac-9ab5-43c6-a235-a7610a34e45b
name: azureml_61b429ac-9ab5-43c6-a235-a7610a34e45b_output_mlflow_log_model_2069815210
path: azureml://subscription

In [16]:
credential = DefaultAzureCredential()

ml_client = MLClient(
    credential,
    subscription_id=subscription_id,
    resource_group_name=resource_group_name,
    workspace_name=ws.name,
)

run_model = Model(
workspace=ws,
path=f"azureml://jobs/{experiment_name}/outputs/artifacts/paths/outputs/",
name="base_model",
description="asdfasdf",
type=AssetTypes.CUSTOM_MODEL
)

ml_client.models.create_or_update(run_model)

WebserviceException: WebserviceException:
	Message: ModelNotFound: Model with name base_model, path azureml://jobs/test/outputs/artifacts/paths/outputs/, description asdfasdf, type custom_model not found in provided workspace
	InnerException None
	ErrorResponse 
{
    "error": {
        "message": "ModelNotFound: Model with name base_model, path azureml://jobs/test/outputs/artifacts/paths/outputs/, description asdfasdf, type custom_model not found in provided workspace"
    }
}

In [27]:
experiment_name = "dispro2_bh"
mlflow.set_experiment(experiment_name)

with mlflow.start_run() as run:
    ml_client = MLClient.from_config(credential=DefaultAzureCredential())
    data_asset = ml_client.data.get("bostonhousing", version="1")

    # Log dataset information
    mlflow.log_param("dataset_name", data_asset.name)
    mlflow.log_param("dataset_version", data_asset.version)


    tbl = mltable.load(f'azureml:/{data_asset.id}')
    pddf_bh = tbl.to_pandas_dataframe()

    # Your training code goes here
    target = "medv"

    X = pddf_bh.drop(target, axis=1)
    y = pddf_bh[target]

    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train the model
    n_estimators = 100
    model = RandomForestRegressor(n_estimators=n_estimators, random_state=42)
    model.fit(X_train, y_train)

    # Evaluate the model
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    print(f"Mean Squared Error: {mse}")
    
    # Log parameters
    mlflow.log_param("n_estimators", n_estimators)
    
    # Log metrics
    mlflow.log_metric("mse_test", mse)
    
    
    # Log the model
    mlflow.sklearn.log_model(model, "model")
    
    # Register the model
    model_uri = f"runs:/{run.info.run_id}/model"
    registered_model = mlflow.register_model(model_uri, "model_bh")


Found the config file in: /config.json
2025/02/26 11:14:26 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'model_bh' already exists. Creating a new version of this model...
2025/02/26 11:14:28 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: model_bh, version 2
Created version '2' of model 'model_bh'.
2025/02/26 11:14:28 INFO mlflow.tracking._tracking_service.client: 🏃 View run good_lobster_lvfz7mfw at: https://switzerlandnorth.api.azureml.ms/mlflow/v2.0/subscriptions/08539d0e-620d-424c-aea8-56853ed8fe3e/resourceGroups/dspro2/providers/Microsoft.MachineLearningServices/workspaces/dspro2ml/#/experiments/f9b76702-65a7-4049-8bf8-767aa494edb5/runs/d7796f66-4182-4e26-9ad0-586cd6a49581.
2025/02/26 11:14:28 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at:

Mean Squared Error: 7.901513892156864


In [23]:
with mlflow.start_run(run_name=experiment_name) as run:
    target = "medv"

    X = pddf_bh.drop(target, axis=1)
    y = pddf_bh[target]

    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train the model
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # Evaluate the model
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    print(f"Mean Squared Error: {mse}")

Mean Squared Error: 7.901513892156864
